# Session 06: Image Segmentation

**CVI4IC - Summer Semester 2026**

Partitioning images into meaningful regions using classical techniques.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

## 1. Thresholding

In [ ]:
# Create a test image with varying intensity
img = np.zeros((256, 256), dtype=np.uint8)
cv2.circle(img, (80, 80), 50, 200, -1)
cv2.circle(img, (180, 180), 60, 120, -1)
cv2.rectangle(img, (30, 160), (100, 230), 80, -1)
img = cv2.GaussianBlur(img, (5, 5), 2)

# Different thresholding methods
_, binary = cv2.threshold(img, 100, 255, cv2.THRESH_BINARY)
_, otsu = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
adaptive = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                  cv2.THRESH_BINARY, 51, 5)

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes[0, 0].imshow(img, cmap="gray")
axes[0, 0].set_title("Original")
axes[0, 1].imshow(binary, cmap="gray")
axes[0, 1].set_title("Binary (thresh=100)")
axes[1, 0].imshow(otsu, cmap="gray")
axes[1, 0].set_title("Otsu's")
axes[1, 1].imshow(adaptive, cmap="gray")
axes[1, 1].set_title("Adaptive")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. K-Means Color Segmentation

In [ ]:
# Create a colorful test image
color_img = np.zeros((200, 200, 3), dtype=np.uint8)
color_img[0:100, 0:100] = [255, 50, 50]
color_img[0:100, 100:200] = [50, 255, 50]
color_img[100:200, 0:100] = [50, 50, 255]
color_img[100:200, 100:200] = [200, 200, 50]
color_img = cv2.GaussianBlur(color_img, (21, 21), 5)

# K-Means
pixels = color_img.reshape(-1, 3).astype(np.float32)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(cv2.cvtColor(color_img, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")

for idx, k in enumerate([2, 4, 8]):
    _, labels, centers = cv2.kmeans(pixels, k, None, criteria, 10,
                                     cv2.KMEANS_RANDOM_CENTERS)
    segmented = centers[labels.flatten()].reshape(color_img.shape).astype(np.uint8)
    axes[idx + 1].imshow(cv2.cvtColor(segmented, cv2.COLOR_BGR2RGB))
    axes[idx + 1].set_title(f"K-Means (k={k})")

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Watershed Segmentation

In [ ]:
# Create overlapping circles
coins = np.zeros((256, 256), dtype=np.uint8)
cv2.circle(coins, (80, 128), 50, 200, -1)
cv2.circle(coins, (170, 128), 50, 200, -1)
cv2.circle(coins, (128, 80), 40, 180, -1)

# Threshold
_, thresh = cv2.threshold(coins, 100, 255, cv2.THRESH_BINARY)

# Distance transform
dist = cv2.distanceTransform(thresh, cv2.DIST_L2, 5)
_, sure_fg = cv2.threshold(dist, 0.5 * dist.max(), 255, 0)
sure_fg = sure_fg.astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(coins, cmap="gray")
axes[0].set_title("Original")
axes[1].imshow(dist, cmap="hot")
axes[1].set_title("Distance Transform")
axes[2].imshow(sure_fg, cmap="gray")
axes[2].set_title("Sure Foreground")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Exercises

1. Apply Otsu's thresholding to a real grayscale image and visualize the histogram
2. Segment a colorful image using K-Means and experiment with different k values
3. Use watershed to separate touching/overlapping objects in a binary image

In [ ]:
# Your code here
